# **Data from** [archive.ics.uci.edu](https://archive.ics.uci.edu/ml/machine-learning-databases/00357/occupancy_data.zip)

In [1]:
import numpy as np
data = np.genfromtxt('datatest.txt', 
                     delimiter=',', 
                     skip_header=1,
                     usecols=(2,3,4,5,6,7))
np.set_printoptions(suppress=True)
#[0] Temperature [1] Humidity [2] Light [3] CO2 [4] HumidityRatio [5] Occupancy
print(data[:3, :])

[[ 23.7         26.272      585.2        749.2          0.00476416
    1.        ]
 [ 23.718       26.29       578.4        760.4          0.00477266
    1.        ]
 [ 23.73        26.23       572.66666667 769.66666667   0.00476515
    1.        ]]


In [ ]:
print(data.shape)
print(data.ndim)

(2665, 6)
2


## A. <u>***Thermal Comfort***</u> 
##### The comfort zone for most individuals typically falls between 20°C to 26°C (68°F to 79°F).

In [ ]:
temp = data[:, 0] #temperature column
total_count = len(temp) 
print(total_count)
comf_zone = (temp >= 20) & (temp <=26) 
comf = temp[comf_zone] #boolean indexing
len(comf)

### Given that all temp (2665) falls within the defined comf_zone, lets apply a stricter zone below!👇

In [ ]:
strict_zone = (temp >=22) & (temp <= 24) #boolean indexing
comf = temp[strict_zone] #boolean indexing
strict_count = len(comf)
print(strict_count)

#percentage of time within strict_zone
percentage = (strict_count / total_count) * 100
print(f'Percentage of time temperature was within comfort zone: {percentage:.2f}%')

## B. <u>***CO2 buildup due to Occupancy***</u>

In [ ]:
co2 = data[:, 3] #co2 column
occupancy = data[:, 5] #occupancy column

#when rooms are occupied
occupied_mask = occupancy >=1 
occupied_times = occupancy[occupied_mask]
empty_times = occupancy[~occupied_mask]

print(len(occupied_times))
print(len(empty_times))
len(occupied_times) + len(empty_times) #ensure all rows were captured

In [ ]:
#CO2 spikes when room is occupied VS when room is not occupied

co2_occupied = co2[occupied_mask]
co2_empty = co2[~occupied_mask]

# average co2 
print(f'avg CO2 when room is occupied is {co2_occupied.mean():.2f} ppm')
print(f'avg CO2 when room is not occupied is {co2_empty.mean():.2f} ppm')

avg_co2_delta = co2_occupied.mean() - co2_empty.mean()
print(f"The CO2 Delta is: {avg_co2_delta:.2f} ppm")

## ***<u>ASHRAE 62.1 Compliance:</u>*** ASHRAE recommends that indoor $CO_2$ levels remain no more than 700 ppm above the ambient outdoor concentration. Assuming an outdoor ambient level of 400 ppm, the limit would be 1100 ppm.

##### ***<u>Current Status:</u>*** At 1014.52 ppm, the room is near borderline. While technically below the 1100 ppm limit, it exceeds the commonly cited 1000 ppm threshold for occupant cognitive performance and comfort.

##### *Also, the calculated delta of 466.90 ppm confirms that occupant metabolic activity is the primary driver of $CO_2$ degradation in this space.*

In [ ]:
#Times co2 exceeded 1100ppm (ASHRAE 62.1)
high_co2_count = (co2 > 1100).sum()
high_co2 = co2[co2 > 1100] #boolean indexing
print(high_co2_count)

high_co2_percentage = (high_co2_count / len(co2)) * 100
print(f'CO2 was above 1100 ppm {high_co2_percentage:.2f} % of the time.')

### Standard Deviation, Max and Min $CO_2$ when rooms are occupied 👇

In [ ]:
# SD, max and min when rooms are occupied
print(f'The SD of CO2 when room is occupied is {co2_occupied.std():.2f} ppm.')
print(f'The max CO2 when room is occupied is {co2_occupied.max():.2f} ppm.')
print(f'The min CO2 when room is occupied is {co2_occupied.min():.2f} ppm.')

#### $CO_2$ Outliers defined by 2 SD away from the mean (occupied rooms only)

In [ ]:
co2_std = co2_occupied.std()
co2_mean = co2_occupied.mean()

co2_upper = co2_mean + (2 * co2_std) #multiply by 2 SD
print(co2_upper)
co2_lower = co2_mean - (2 * co2_std)
print(co2_lower)

outlier_mask = (co2_occupied > co2_upper) | (co2_occupied < co2_lower)
co2_outliers = co2_occupied[outlier_mask]
print(f'Number of 2-Sigma outliers: {outlier_mask.sum()}')

## C. <u>***Energy Waste Indicator using Light***</u>

In [ ]:
light = data[:, 2] #lighting column

light_on = (light >= 300) #lux 
print(light_on.sum()) #number of times light is turned on

#check number of time when light is on when no occupant
light_waste = (light >= 300) & (~occupied_mask)
print(light_waste.sum())

In [ ]:
#percentage of light wastage
waste_percentage = (light_waste.sum() / light_on.sum()) * 100
print(f'Lights were on for {light_on.sum()} minutes total.')
print(f'Out of that, {waste_percentage:.2f}% of the energy was wasted on empty rooms.')